In [1]:
#fill NA with None
from sklearn import tree
from sklearn.preprocessing import LabelEncoder
import pandas as pd
import random
import numpy as np
from sklearn.metrics import accuracy_score
from sklearn.model_selection import KFold
from statistics import mean
from sklearn.naive_bayes import ComplementNB

In [2]:
#https://stackoverflow.com/questions/20224526/how-to-extract-the-decision-rules-from-scikit-learn-decision-tree
def get_code(tree, feature_names):
        left      = tree.tree_.children_left
        right     = tree.tree_.children_right
        threshold = tree.tree_.threshold
        features  = [feature_names[i] for i in tree.tree_.feature]
        value = tree.tree_.value

        def recurse(left, right, threshold, features, node):
                if (threshold[node] != -2):
                        print("if ( " + features[node] + " <= " + str(threshold[node]) + " ) {")
                        if left[node] != -1:
                                recurse (left, right, threshold, features,left[node])
                        print( "} else {")
                        if right[node] != -1:
                                recurse (left, right, threshold, features,right[node])
                        print( "}")
                else:
                        print( "return " + str(value[node]))

        recurse(left, right, threshold, features, 0)

In [3]:
data = pd.read_excel('dataset_gen.xlsx') 
dataset_all_gen = pd.read_excel('dataset_all_gen.xlsx')
dataset_all_original = pd.read_excel('dataset_all_original.xlsx')
dataset_jaccard_gen = pd.read_excel('dataset_jaccard_gen.xlsx')
dataset_jaccard_original = pd.read_excel('dataset_jaccard_original.xlsx')
dataset_myroom_gen = pd.read_excel('dataset_myroom_gen.xlsx')
dataset_myroom_original = pd.read_excel('dataset_myroom_original.xlsx')
dataset_university_gen = pd.read_excel('dataset_university_gen.xlsx')
dataset_university_original = pd.read_excel('dataset_university_original.xlsx')

In [4]:
data = data.sample(frac = 1) 
dataset_all_gen = dataset_all_gen.sample(frac = 1) 
dataset_all_original = dataset_all_original.sample(frac = 1) 
dataset_jaccard_gen = dataset_jaccard_gen.sample(frac = 1) 
dataset_jaccard_original = dataset_jaccard_original.sample(frac = 1) 
dataset_myroom_gen = dataset_myroom_gen.sample(frac = 1) 
dataset_myroom_original = dataset_myroom_original.sample(frac = 1) 
dataset_university_gen = dataset_university_gen.sample(frac = 1) 
dataset_university_original = dataset_university_original.sample(frac = 1) 

In [5]:

data = data.reset_index(drop=True)
dataset_all_gen = dataset_all_gen.reset_index(drop=True)
dataset_all_original = dataset_all_original.reset_index(drop=True)
dataset_jaccard_gen = dataset_jaccard_gen.reset_index(drop=True)
dataset_jaccard_original = dataset_jaccard_original.reset_index(drop=True)
dataset_myroom_gen = dataset_myroom_gen.reset_index(drop=True)
dataset_myroom_original = dataset_myroom_original.reset_index(drop=True)
dataset_university_gen = dataset_university_gen.reset_index(drop=True)
dataset_university_original = dataset_university_original.reset_index(drop=True)

In [6]:
data.isnull().sum()

type             0
Name             0
Vendor           0
DNS           2304
mDNS          2704
NUPNP        11000
open_port     1731
SSDP_Type     2304
SSDP_t       11000
SSDP_v       11000
dtype: int64

In [7]:
data = data.fillna("NaN")

In [8]:
#fill Na
dataset_all_gen = dataset_all_gen.fillna("NaN")
dataset_all_original = dataset_all_original.fillna("NaN")
dataset_jaccard_gen = dataset_jaccard_gen.fillna("NaN")
dataset_jaccard_original = dataset_jaccard_original.fillna("NaN")
dataset_myroom_gen = dataset_myroom_gen.fillna("NaN")
dataset_myroom_original = dataset_myroom_original.fillna("NaN")
dataset_university_gen = dataset_university_gen.fillna("NaN")
dataset_university_original = dataset_university_original.fillna("NaN")

In [9]:
data.isnull().sum()

type         0
Name         0
Vendor       0
DNS          0
mDNS         0
NUPNP        0
open_port    0
SSDP_Type    0
SSDP_t       0
SSDP_v       0
dtype: int64

In [10]:
data.dtypes

type         object
Name         object
Vendor       object
DNS          object
mDNS         object
NUPNP        object
open_port    object
SSDP_Type    object
SSDP_t       object
SSDP_v       object
dtype: object

In [11]:
data = data.astype('category')

In [12]:
dataset_all_gen = dataset_all_gen.astype('category')
dataset_all_original = dataset_all_original.astype('category')
dataset_jaccard_gen = dataset_jaccard_gen.astype('category')
dataset_jaccard_original = dataset_jaccard_original.astype('category')
dataset_myroom_gen = dataset_myroom_gen.astype('category')
dataset_myroom_original = dataset_myroom_original.astype('category')
dataset_university_gen = dataset_university_gen.astype('category')
dataset_university_original = dataset_university_original.astype('category')

In [13]:
data.dtypes

type         category
Name         category
Vendor       category
DNS          category
mDNS         category
NUPNP        category
open_port    category
SSDP_Type    category
SSDP_t       category
SSDP_v       category
dtype: object

In [14]:
data

,type,Name,Vendor,DNS,mDNS,NUPNP,open_port,SSDP_Type,SSDP_t,SSDP_v
0,Energy Management,Belkin Wemo switch,Belkin International Inc.,dailymotion.com,_gembucket._udp.local.,NaN,tcp/1883,NOTIFY * HTTP/1.1,NaN,NaN
1,Non-IOT,Macbook1,"Apple, Inc.",networksolutions.com,NaN,NaN,tcp/1900,M-SEARCH * HTTP/1.1,NaN,NaN
2,Appliances,Google Home,Google,unesco.org,_fixflex._udp.local.,NaN,tcp/80,NOTIFY * HTTP/1.1,NaN,NaN
3,Cameras,"Insteon Camera, wifi",Insteon,nymag.com,_zathin._udp.local.,NaN,tcp/443,NOTIFY * HTTP/1.1,NaN,NaN
4,Appliances,Bezeq smarthome,Bezeq Group,google.com.hk,NaN,NaN,tcp/22,NOTIFY * HTTP/1.1,NaN,NaN
5,Appliances,Roomba,iRobot Corporation,mac.com,NaN,NaN,tcp/1900,NOTIFY * HTTP/1.1,NaN,NaN
6,Non-IOT,Nexus 5x Smartphone,LG Electronics,japanpost.jp,_lotlux._udp.local.,NaN,tcp/1900,M-SEARCH * HTTP/1.1,NaN,NaN
7,Non-IOT,IPhone,"Apple, Inc.",vkontakte.ru,NaN,NaN,tcp/1883,M-SEARCH * HTTP/1.1,NaN,NaN
8,Hubs,Logitech Harmony Hub,Logitech,mapquest.com,_solarbreeze._tcp.local.,NaN,tcp/1900,NOTIFY * HTTP/1.1,NaN,NaN
9,Cameras,"Insteon Camera, wifi",Insteon,about.com,_konklab._udp.local.,NaN,NaN,NOTIFY * HTTP/1.1,NaN,NaN


In [15]:
y = data.drop_duplicates(keep='first')
y = y.drop(['Name'], axis=1)
y = y.drop_duplicates(keep='first')
# y = y.reset_index()
# shuffle the DataFrame rows
# y = y.sample(frac = 1) 


In [16]:
len(y)

10610

In [17]:
#drop_duplicates
dataset_all_gen_y = dataset_all_gen.drop_duplicates(keep='first')
dataset_all_original_y = dataset_all_original.drop_duplicates(keep='first')
dataset_jaccard_gen_y = dataset_jaccard_gen.drop_duplicates(keep='first')
dataset_jaccard_original_y = dataset_jaccard_original.drop_duplicates(keep='first')
dataset_myroom_gen_y = dataset_myroom_gen.drop_duplicates(keep='first')
dataset_myroom_original_y = dataset_myroom_original.drop_duplicates(keep='first')
dataset_university_gen_y = dataset_university_gen.drop_duplicates(keep='first')
dataset_university_original_y = dataset_university_original.drop_duplicates(keep='first')

In [18]:
#drop name
dataset_all_gen_y = dataset_all_gen_y.drop(['Name'], axis=1)
dataset_all_original_y = dataset_all_original_y.drop(['Name'], axis=1)
dataset_jaccard_gen_y = dataset_jaccard_gen_y.drop(['Name'], axis=1)
dataset_jaccard_original_y = dataset_jaccard_original_y.drop(['Name'], axis=1)
dataset_myroom_gen_y = dataset_myroom_gen_y.drop(['Name'], axis=1)
dataset_myroom_original_y = dataset_myroom_original_y.drop(['Name'], axis=1)
dataset_university_gen_y = dataset_university_gen_y.drop(['Name'], axis=1)
dataset_university_original_y = dataset_university_original_y.drop(['Name'], axis=1)

In [19]:
#drop_duplicates
dataset_all_gen_y = dataset_all_gen_y.drop_duplicates(keep='first')
dataset_all_original_y = dataset_all_original_y.drop_duplicates(keep='first')
dataset_jaccard_gen_y = dataset_jaccard_gen_y.drop_duplicates(keep='first')
dataset_jaccard_original_y = dataset_jaccard_original_y.drop_duplicates(keep='first')
dataset_myroom_gen_y = dataset_myroom_gen_y.drop_duplicates(keep='first')
dataset_myroom_original_y = dataset_myroom_original_y.drop_duplicates(keep='first')
dataset_university_gen_y = dataset_university_gen_y.drop_duplicates(keep='first')
dataset_university_original_y = dataset_university_original_y.drop_duplicates(keep='first')

In [20]:
y = y.apply(LabelEncoder().fit_transform)

In [21]:
dataset_all_gen_y = dataset_all_gen_y.apply(lambda col: LabelEncoder().fit_transform(col.astype(str)), axis=0, result_type='expand')
dataset_all_original_y = dataset_all_original_y.apply(lambda col: LabelEncoder().fit_transform(col.astype(str)), axis=0, result_type='expand')
dataset_jaccard_gen_y = dataset_jaccard_gen_y.apply(lambda col: LabelEncoder().fit_transform(col.astype(str)), axis=0, result_type='expand')
dataset_jaccard_original_y = dataset_jaccard_original_y.apply(lambda col: LabelEncoder().fit_transform(col.astype(str)), axis=0, result_type='expand')
dataset_myroom_gen_y = dataset_myroom_gen_y.apply(lambda col: LabelEncoder().fit_transform(col.astype(str)), axis=0, result_type='expand')
dataset_myroom_original_y = dataset_myroom_original_y.apply(lambda col: LabelEncoder().fit_transform(col.astype(str)), axis=0, result_type='expand')
dataset_university_gen_y = dataset_university_gen_y.apply(lambda col: LabelEncoder().fit_transform(col.astype(str)), axis=0, result_type='expand')
dataset_university_original_y = dataset_university_original_y.apply(lambda col: LabelEncoder().fit_transform(col.astype(str)), axis=0, result_type='expand')

In [22]:
list_data = []
list_data.append(y)
list_data.append(dataset_all_gen_y)
list_data.append(dataset_all_original_y)
list_data.append(dataset_jaccard_gen_y)
list_data.append(dataset_jaccard_original_y)
list_data.append(dataset_myroom_gen_y)
list_data.append(dataset_myroom_original_y)
list_data.append(dataset_university_gen_y)
list_data.append(dataset_university_original_y)
len(list_data)

9

In [23]:
accuracy = []
accuracy_t = []
accuracy_temp = []

In [24]:
for data in list_data:
    for i in range(100):
        kf = KFold(n_splits=10, random_state=None, shuffle=True)
#         kf.get_n_splits(data)
        for train_index, test_index in kf.split(data):
        #     print(index)
#             print("TRAIN:", train_index, "TEST:", test_index)
            train, test = data.iloc[train_index], data.iloc[test_index]
#             print("TRAIN:", train, "TEST:", test)
        #     print(len(train))
#             print("TRAIN:", len(train), "TEST:", len(test))
            label_data = train.type
            train_data = train.drop(['type'], axis=1)
            tlabel_data = test.type
            test_data = test.drop(['type'], axis=1)
            truelabel_data = pd.DataFrame(tlabel_data)
            truelabel_data = truelabel_data.reset_index(drop=True)
            clf = ComplementNB()
#             clf = tree.DecisionTreeClassifier()
#             clf = clf.fit(train_data,label_data )
            clf = clf.fit(train_data,label_data )
#             predict = clf.predict(test_data)
            predict = clf.predict(test_data)
#             print(truelabel_data)
#             print(predict)
        #     len(predict)
            result_label_data = pd.DataFrame(columns = ['type'])
            result_label_data.type = predict
            compare = pd.DataFrame(columns = ['truelabel_data','predictedlabel'])
            compare.truelabel_data = truelabel_data.type
            compare.predictedlabel = result_label_data.type
#             print(compare)
            t = (accuracy_score(compare.truelabel_data, compare.predictedlabel))
            accuracy_temp.append(t)
            print(i,"  Accuracy: ",t)
        accuracy_t.append(mean(accuracy_temp))
    accuracy.append(mean(accuracy_t))

0   Accuracy:  0.24222431668237512
0   Accuracy:  0.25164938737040526
0   Accuracy:  0.27521206409048066
0   Accuracy:  0.2648444863336475
0   Accuracy:  0.26578699340245054
0   Accuracy:  0.26107445805843543
0   Accuracy:  0.28180961357210177
0   Accuracy:  0.2639019792648445
0   Accuracy:  0.27521206409048066
0   Accuracy:  0.295004712535344
1   Accuracy:  0.26201696512723843
1   Accuracy:  0.26767200754005654
1   Accuracy:  0.2535344015080113
1   Accuracy:  0.2761545711592837
1   Accuracy:  0.2629594721960415
1   Accuracy:  0.2855796418473139
1   Accuracy:  0.2855796418473139
1   Accuracy:  0.2836946277097078
1   Accuracy:  0.26013195098963243
1   Accuracy:  0.2412818096135721
2   Accuracy:  0.2884071630537229
2   Accuracy:  0.22431668237511782
2   Accuracy:  0.24505183788878418
2   Accuracy:  0.27898209236569277
2   Accuracy:  0.2761545711592837
2   Accuracy:  0.27521206409048066
2   Accuracy:  0.27332704995287466
2   Accuracy:  0.27332704995287466
2   Accuracy:  0.2723845428840716

23   Accuracy:  0.2836946277097078
23   Accuracy:  0.25918944392082943
23   Accuracy:  0.26013195098963243
23   Accuracy:  0.23939679547596607
23   Accuracy:  0.2874646559849199
23   Accuracy:  0.2836946277097078
24   Accuracy:  0.28086710650329877
24   Accuracy:  0.292177191328935
24   Accuracy:  0.2525918944392083
24   Accuracy:  0.24410933081998115
24   Accuracy:  0.28086710650329877
24   Accuracy:  0.2535344015080113
24   Accuracy:  0.2695570216776626
24   Accuracy:  0.27992459943449577
24   Accuracy:  0.24976437323279924
24   Accuracy:  0.2770970782280867
25   Accuracy:  0.2629594721960415
25   Accuracy:  0.26201696512723843
25   Accuracy:  0.26013195098963243
25   Accuracy:  0.2780395852968897
25   Accuracy:  0.2714420358152686
25   Accuracy:  0.2554194156456173
25   Accuracy:  0.26861451460885954
25   Accuracy:  0.26201696512723843
25   Accuracy:  0.27332704995287466
25   Accuracy:  0.26767200754005654
26   Accuracy:  0.25918944392082943
26   Accuracy:  0.25070688030160226
26   

48   Accuracy:  0.28180961357210177
48   Accuracy:  0.2403393025447691
48   Accuracy:  0.26107445805843543
48   Accuracy:  0.2865221489161169
48   Accuracy:  0.24316682375117812
48   Accuracy:  0.29688972667295005
48   Accuracy:  0.2648444863336475
48   Accuracy:  0.2704995287464656
48   Accuracy:  0.26767200754005654
48   Accuracy:  0.2770970782280867
49   Accuracy:  0.27426955702167766
49   Accuracy:  0.2525918944392083
49   Accuracy:  0.24599434495758718
49   Accuracy:  0.26013195098963243
49   Accuracy:  0.2846371347785108
49   Accuracy:  0.293119698397738
49   Accuracy:  0.2629594721960415
49   Accuracy:  0.27426955702167766
49   Accuracy:  0.24505183788878418
49   Accuracy:  0.2770970782280867
50   Accuracy:  0.25918944392082943
50   Accuracy:  0.2535344015080113
50   Accuracy:  0.26767200754005654
50   Accuracy:  0.26578699340245054
50   Accuracy:  0.2639019792648445
50   Accuracy:  0.2855796418473139
50   Accuracy:  0.2780395852968897
50   Accuracy:  0.24599434495758718
50   Ac

71   Accuracy:  0.2648444863336475
71   Accuracy:  0.2846371347785108
71   Accuracy:  0.26013195098963243
71   Accuracy:  0.24505183788878418
71   Accuracy:  0.26201696512723843
71   Accuracy:  0.25918944392082943
71   Accuracy:  0.26107445805843543
71   Accuracy:  0.2648444863336475
72   Accuracy:  0.26861451460885954
72   Accuracy:  0.2582469368520264
72   Accuracy:  0.2827521206409048
72   Accuracy:  0.2780395852968897
72   Accuracy:  0.2573044297832234
72   Accuracy:  0.2997172478793591
72   Accuracy:  0.2884071630537229
72   Accuracy:  0.28180961357210177
72   Accuracy:  0.23751178133836004
72   Accuracy:  0.2639019792648445
73   Accuracy:  0.2770970782280867
73   Accuracy:  0.26013195098963243
73   Accuracy:  0.26201696512723843
73   Accuracy:  0.2761545711592837
73   Accuracy:  0.24599434495758718
73   Accuracy:  0.2865221489161169
73   Accuracy:  0.27521206409048066
73   Accuracy:  0.2770970782280867
73   Accuracy:  0.24222431668237512
73   Accuracy:  0.28934967012252594
74   A

94   Accuracy:  0.2582469368520264
95   Accuracy:  0.2535344015080113
95   Accuracy:  0.2573044297832234
95   Accuracy:  0.27898209236569277
95   Accuracy:  0.2836946277097078
95   Accuracy:  0.2563619227144204
95   Accuracy:  0.2827521206409048
95   Accuracy:  0.27426955702167766
95   Accuracy:  0.2525918944392083
95   Accuracy:  0.2535344015080113
95   Accuracy:  0.2874646559849199
96   Accuracy:  0.2761545711592837
96   Accuracy:  0.2573044297832234
96   Accuracy:  0.27992459943449577
96   Accuracy:  0.26107445805843543
96   Accuracy:  0.2714420358152686
96   Accuracy:  0.2761545711592837
96   Accuracy:  0.2582469368520264
96   Accuracy:  0.2761545711592837
96   Accuracy:  0.2629594721960415
96   Accuracy:  0.2836946277097078
97   Accuracy:  0.2704995287464656
97   Accuracy:  0.236569274269557
97   Accuracy:  0.2827521206409048
97   Accuracy:  0.26861451460885954
97   Accuracy:  0.28180961357210177
97   Accuracy:  0.27426955702167766
97   Accuracy:  0.27992459943449577
97   Accuracy

19   Accuracy:  0.2816651904340124
19   Accuracy:  0.2969858156028369
19   Accuracy:  0.2907801418439716
19   Accuracy:  0.2526595744680851
20   Accuracy:  0.2674933569530558
20   Accuracy:  0.2630646589902569
20   Accuracy:  0.2683790965456156
20   Accuracy:  0.2674933569530558
20   Accuracy:  0.2869796279893711
20   Accuracy:  0.2657218777679362
20   Accuracy:  0.2807794508414526
20   Accuracy:  0.27925531914893614
20   Accuracy:  0.2774822695035461
20   Accuracy:  0.274822695035461
21   Accuracy:  0.2524357838795394
21   Accuracy:  0.27192205491585475
21   Accuracy:  0.27989371124889284
21   Accuracy:  0.26217891939769705
21   Accuracy:  0.2736935341009743
21   Accuracy:  0.29317980513728964
21   Accuracy:  0.27989371124889284
21   Accuracy:  0.28812056737588654
21   Accuracy:  0.2765957446808511
21   Accuracy:  0.26152482269503546
22   Accuracy:  0.2683790965456156
22   Accuracy:  0.254207263064659
22   Accuracy:  0.28255093002657217
22   Accuracy:  0.2843224092116918
22   Accuracy

... (27 outputs left out)

13   Accuracy:  0.4827586206896552
13   Accuracy:  0.4827586206896552
13   Accuracy:  0.3448275862068966
13   Accuracy:  0.3793103448275862
13   Accuracy:  0.39285714285714285
13   Accuracy:  0.2857142857142857
13   Accuracy:  0.39285714285714285
13   Accuracy:  0.5
13   Accuracy:  0.5714285714285714
13   Accuracy:  0.5357142857142857
14   Accuracy:  0.3448275862068966
14   Accuracy:  0.4827586206896552
14   Accuracy:  0.4827586206896552
14   Accuracy:  0.5862068965517241
14   Accuracy:  0.35714285714285715
14   Accuracy:  0.35714285714285715
14   Accuracy:  0.39285714285714285
14   Accuracy:  0.5
14   Accuracy:  0.42857142857142855
14   Accuracy:  0.35714285714285715
15   Accuracy:  0.5172413793103449
15   Accuracy:  0.1724137931034483
15   Accuracy:  0.41379310344827586
15   Accuracy:  0.5517241379310345
15   Accuracy:  0.32142857142857145
15   Accuracy:  0.39285714285714285
15   Accuracy:  0.5357142857142857
15   Accuracy:  0.4642857142857143
15   Accuracy:  0.35714285714285715
15  

37   Accuracy:  0.39285714285714285
37   Accuracy:  0.32142857142857145
37   Accuracy:  0.4642857142857143
37   Accuracy:  0.42857142857142855
37   Accuracy:  0.4642857142857143
37   Accuracy:  0.39285714285714285
38   Accuracy:  0.5517241379310345
38   Accuracy:  0.3448275862068966
38   Accuracy:  0.3448275862068966
38   Accuracy:  0.4482758620689655
38   Accuracy:  0.2857142857142857
38   Accuracy:  0.35714285714285715
38   Accuracy:  0.32142857142857145
38   Accuracy:  0.4642857142857143
38   Accuracy:  0.6071428571428571
38   Accuracy:  0.42857142857142855
39   Accuracy:  0.3793103448275862
39   Accuracy:  0.3793103448275862
39   Accuracy:  0.4482758620689655
39   Accuracy:  0.4482758620689655
39   Accuracy:  0.39285714285714285
39   Accuracy:  0.6428571428571429
39   Accuracy:  0.39285714285714285
39   Accuracy:  0.35714285714285715
39   Accuracy:  0.5714285714285714
39   Accuracy:  0.39285714285714285
40   Accuracy:  0.4482758620689655
40   Accuracy:  0.3103448275862069
40   Accu

61   Accuracy:  0.35714285714285715
61   Accuracy:  0.32142857142857145
61   Accuracy:  0.39285714285714285
61   Accuracy:  0.42857142857142855
61   Accuracy:  0.4642857142857143
61   Accuracy:  0.42857142857142855
62   Accuracy:  0.4482758620689655
62   Accuracy:  0.41379310344827586
62   Accuracy:  0.4827586206896552
62   Accuracy:  0.6206896551724138
62   Accuracy:  0.42857142857142855
62   Accuracy:  0.39285714285714285
62   Accuracy:  0.35714285714285715
62   Accuracy:  0.35714285714285715
62   Accuracy:  0.32142857142857145
62   Accuracy:  0.42857142857142855
63   Accuracy:  0.3103448275862069
63   Accuracy:  0.4827586206896552
63   Accuracy:  0.5517241379310345
63   Accuracy:  0.3793103448275862
63   Accuracy:  0.35714285714285715
63   Accuracy:  0.42857142857142855
63   Accuracy:  0.5
63   Accuracy:  0.32142857142857145
63   Accuracy:  0.39285714285714285
63   Accuracy:  0.4642857142857143
64   Accuracy:  0.5172413793103449
64   Accuracy:  0.5172413793103449
64   Accuracy:  0.6

85   Accuracy:  0.5357142857142857
86   Accuracy:  0.41379310344827586
86   Accuracy:  0.6206896551724138
86   Accuracy:  0.4482758620689655
86   Accuracy:  0.3448275862068966
86   Accuracy:  0.35714285714285715
86   Accuracy:  0.42857142857142855
86   Accuracy:  0.42857142857142855
86   Accuracy:  0.21428571428571427
86   Accuracy:  0.39285714285714285
86   Accuracy:  0.4642857142857143
87   Accuracy:  0.5172413793103449
87   Accuracy:  0.3103448275862069
87   Accuracy:  0.3448275862068966
87   Accuracy:  0.4482758620689655
87   Accuracy:  0.4642857142857143
87   Accuracy:  0.5357142857142857
87   Accuracy:  0.39285714285714285
87   Accuracy:  0.39285714285714285
87   Accuracy:  0.39285714285714285
87   Accuracy:  0.39285714285714285
88   Accuracy:  0.5517241379310345
88   Accuracy:  0.5172413793103449
88   Accuracy:  0.4482758620689655
88   Accuracy:  0.27586206896551724
88   Accuracy:  0.32142857142857145
88   Accuracy:  0.5
88   Accuracy:  0.2857142857142857
88   Accuracy:  0.42857

In [25]:
print(len(y),"\t Accuracy for dataset_gen: \t", accuracy[0])
print(len(dataset_all_gen_y),"\t Accuracy for dataset_all_gen: \t", accuracy[1])
print(len(dataset_all_original_y),"\t Accuracy for dataset_all_original: \t", accuracy[2])
print(len(dataset_jaccard_gen_y),"\t Accuracy for dataset_jaccard_gen: \t", accuracy[3])
print(len(dataset_jaccard_original_y),"\t Accuracy for dataset_jaccard_original: \t", accuracy[4])
print(len(dataset_myroom_gen_y),"\t Accuracy for dataset_myroom_gen: \t", accuracy[5])
print(len(dataset_myroom_original_y),"\t Accuracy for dataset_myroom_original: \t", accuracy[6])
print(len(dataset_university_gen_y),"\t Accuracy for dataset_university_gen: \t", accuracy[7])
print(len(dataset_university_original_y),"\t Accuracy for dataset_university_original: \t", accuracy[8])

10610 	 Accuracy for dataset_gen: 	 0.26820461382945693
11287 	 Accuracy for dataset_all_gen: 	 0.269132891632083
677 	 Accuracy for dataset_all_original: 	 0.28247656558236767
10912 	 Accuracy for dataset_jaccard_gen: 	 0.2937090962841348
302 	 Accuracy for dataset_jaccard_original: 	 0.3104326189946938
10701 	 Accuracy for dataset_myroom_gen: 	 0.32732361090767986
91 	 Accuracy for dataset_myroom_original: 	 0.3417436608522979
10894 	 Accuracy for dataset_university_gen: 	 0.3542543592025939
284 	 Accuracy for dataset_university_original: 	 0.3627685586731122
